In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

df = pd.read_excel("Data_Model_IoTMLCQ_2024.xlsx")

X = df[['Temperature (°C)', 'pH', 'Turbidity (NTU)']]
y = df['Dissolved Oxygen (mg/L)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("R2 :", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))

joblib.dump(model, "do_model.pkl")

R2 : 0.9996133933881711
MAE: 0.0012844618696326298


['do_model.pkl']

In [3]:
print(X.columns)

Index(['Temperature (°C)', 'pH', 'Turbidity (NTU)'], dtype='str')


In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
train_size = int(0.8 * len(df))

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

In [6]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring='r2'
)

print(scores)
print(scores.mean())

[-132.04141944    0.97711976    0.97969906    0.68736924   -9.87010416]
-27.853467106636952


In [7]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

scores = cross_val_score(
    model,
    X,
    y,
    cv=tscv,
    scoring='r2'
)

print(scores)

[-2.59927923e-01  2.57642095e-01 -2.01123711e-01 -2.05196631e-01
 -6.75000000e+02]


In [8]:
print(df.describe())

                  Datetime  Average Fish Weight (g)  Survival Rate (%)  \
count                 4383              4383.000000        4383.000000   
mean   2024-04-01 07:00:00               273.391088          93.857269   
min    2024-01-01 00:00:00               262.050000          92.850000   
25%    2024-02-15 15:30:00               262.820000          92.930000   
50%    2024-04-01 07:00:00               275.820000          92.960000   
75%    2024-05-16 22:30:00               278.100000          95.270000   
max    2024-07-01 14:00:00               286.500000          96.300000   
std                    NaN                 8.675050           1.337427   

       Disease Occurrence (Cases)  Temperature (°C)  Dissolved Oxygen (mg/L)  \
count                 4383.000000       4383.000000              4383.000000   
mean                     1.159480         27.344908                 6.929651   
min                      1.000000         26.500000                 6.340000   
25%          

In [9]:
print(df.columns.tolist())

['Datetime', 'Month', 'Average Fish Weight (g)', 'Survival Rate (%)', 'Disease Occurrence (Cases)', 'Temperature (°C)', 'Dissolved Oxygen (mg/L)', 'pH', 'Turbidity (NTU)', 'Month_Num', 'month_x', 'Oxygenation Interventions', 'Corrective Interventions', 'Average Temperature (°C)', 'High Temperature (°C)', 'Low Temperature (°C)', 'Precipitation (inches)', 'month_y', 'day', 'hour', 'oxigeno_scaled', 'ph', 'turbidez', 'Oxygenation Automatic', 'Corrective Measures', 'Thermal Risk Index', 'Low Oxygen Alert', 'Health Status']


In [11]:
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np

# ============================
# 1. LOAD DATASET
# ============================
df = pd.read_excel("Data_Model_IoTMLCQ_2024.xlsx")

# Urutkan berdasarkan waktu
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime')

# ============================
# 2. PILIH FITUR
# ============================
X = df[
    [
        'Temperature (°C)',
        'pH',
        'Turbidity (NTU)'
    ]
]

y = df['Dissolved Oxygen (mg/L)']

print("Fitur:")
print(X.columns)

# ============================
# 3. TRAIN TEST SPLIT
# Karena data time series,
# jangan pakai shuffle.
# ============================
train_size = int(len(df) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

# ============================
# 4. TRAIN MODEL
# ============================
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# ============================
# 5. PREDIKSI
# ============================
y_pred = model.predict(X_test)

# ============================
# 6. EVALUASI
# ============================
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n===== HASIL EVALUASI =====")
print("R²   :", r2)
print("MAE  :", mae)
print("RMSE :", rmse)

# ============================
# 7. CROSS VALIDATION
# ============================
tscv = TimeSeriesSplit(n_splits=5)

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=tscv,
    scoring='r2'
)

print("\n===== CROSS VALIDATION =====")
print(cv_scores)
print("Mean R² :", cv_scores.mean())

# ============================
# 8. FEATURE IMPORTANCE
# ============================
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
})

print("\n===== FEATURE IMPORTANCE =====")
print(
    importance.sort_values(
        by='Importance',
        ascending=False
    )
)

# ============================
# 9. SIMPAN MODEL
# ============================
joblib.dump(model, 'do_model.pkl')

print("\nModel berhasil disimpan.")

Fitur:
Index(['Temperature (°C)', 'pH', 'Turbidity (NTU)'], dtype='str')

===== HASIL EVALUASI =====
R²   : -10.014707646918586
MAE  : 0.8164215024703545
RMSE : 0.8385172381432257

===== CROSS VALIDATION =====
[-0.26007392  0.25544466 -0.189209   -0.18822885  0.        ]
Mean R² : -0.07641342268358167

===== FEATURE IMPORTANCE =====
            Feature  Importance
0  Temperature (°C)    0.957300
1                pH    0.034678
2   Turbidity (NTU)    0.008022

Model berhasil disimpan.


In [12]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_excel("Data_Model_IoTMLCQ_2024.xlsx")

# Pastikan data terurut berdasarkan waktu
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime')

# Target
y = df['Dissolved Oxygen (mg/L)']

# =========================
# 2. KOMBINASI FITUR
# =========================
feature_sets = {
    'Sensor_IoT': [
        'Temperature (°C)',
        'pH',
        'Turbidity (NTU)'
    ],

    'Sensor_IoT_Waktu': [
        'Temperature (°C)',
        'pH',
        'Turbidity (NTU)',
        'Month_Num',
        'day',
        'hour'
    ],

    'Dataset_Terbaik': [
        'Temperature (°C)',
        'pH',
        'Turbidity (NTU)',
        'Month_Num',
        'Survival Rate (%)',
        'Disease Occurrence (Cases)'
    ],

    'Semua_Tanpa_Leakage': [
        'Temperature (°C)',
        'pH',
        'Turbidity (NTU)',
        'Month_Num',
        'day',
        'hour',
        'Average Fish Weight (g)',
        'Survival Rate (%)',
        'Disease Occurrence (Cases)'
    ]
}

# =========================
# 3. CROSS VALIDATION
# =========================
tscv = TimeSeriesSplit(n_splits=5)

hasil = []

for nama, fitur in feature_sets.items():

    X = df[fitur]

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=tscv,
        scoring='r2'
    )

    hasil.append({
        'Model': nama,
        'Mean_R2': scores.mean(),
        'Std_R2': scores.std()
    })

    print("=" * 60)
    print("Model :", nama)
    print("Fitur :", fitur)
    print("R2 per Fold :", scores)
    print("Mean R2 :", scores.mean())
    print()

# =========================
# 4. TABEL HASIL
# =========================
hasil_df = pd.DataFrame(hasil)
hasil_df = hasil_df.sort_values(
    by='Mean_R2',
    ascending=False
)

print("\n===== HASIL AKHIR =====")
print(hasil_df)

Model : Sensor_IoT
Fitur : ['Temperature (°C)', 'pH', 'Turbidity (NTU)']
R2 per Fold : [-0.26007392  0.25544466 -0.189209   -0.18822885  0.        ]
Mean R2 : -0.07641342268358293

Model : Sensor_IoT_Waktu
Fitur : ['Temperature (°C)', 'pH', 'Turbidity (NTU)', 'Month_Num', 'day', 'hour']
R2 per Fold : [-2.07663389e-01 -3.63178815e-03  5.94827468e-02 -1.56118414e-01
 -1.41074309e+28]
Mean R2 : -2.821486180151233e+27

Model : Dataset_Terbaik
Fitur : ['Temperature (°C)', 'pH', 'Turbidity (NTU)', 'Month_Num', 'Survival Rate (%)', 'Disease Occurrence (Cases)']
R2 per Fold : [-0.2059309   0.27610153  0.08931787 -0.33217685  0.        ]
Mean R2 : -0.034537670624846584

Model : Semua_Tanpa_Leakage
Fitur : ['Temperature (°C)', 'pH', 'Turbidity (NTU)', 'Month_Num', 'day', 'hour', 'Average Fish Weight (g)', 'Survival Rate (%)', 'Disease Occurrence (Cases)']
R2 per Fold : [-2.00283211e-01  6.03812891e-01 -2.36946902e-01 -3.98022992e-01
 -5.50265611e+27]
Mean R2 : -1.1005312228361867e+27


===== HAS

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except:
    xgb_available = False


# ==========================================
# LOAD DATA
# ==========================================
df = pd.read_excel("Data_Model_IoTMLCQ_2024.xlsx")

df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime')


# ==========================================
# FEATURE ENGINEERING
# ==========================================

# interaksi fitur
df['temp_ph'] = (
    df['Temperature (°C)']
    * df['pH']
)

df['temp2'] = (
    df['Temperature (°C)'] ** 2
)

df['ph2'] = (
    df['pH'] ** 2
)

df['turb2'] = (
    df['Turbidity (NTU)'] ** 2
)

# lag feature
df['temp_prev'] = (
    df['Temperature (°C)']
    .shift(1)
)

df['ph_prev'] = (
    df['pH']
    .shift(1)
)

df['turb_prev'] = (
    df['Turbidity (NTU)']
    .shift(1)
)

df = df.dropna()


# ==========================================
# PILIH FITUR
# ==========================================
features = [
    'Temperature (°C)',
    'pH',
    'Turbidity (NTU)',

    'Month_Num',
    'day',
    'hour',

    'temp_ph',
    'temp2',
    'ph2',
    'turb2',

    'temp_prev',
    'ph_prev',
    'turb_prev'
]

X = df[features]
y = df['Dissolved Oxygen (mg/L)']


# ==========================================
# TRAIN TEST SPLIT
# ==========================================
train_size = int(len(df) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]


# ==========================================
# EVALUASI
# ==========================================
hasil = []


def evaluate_model(name, model):

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(
        mean_squared_error(y_test, pred)
    )

    hasil.append({
        'Model': name,
        'R2': r2,
        'MAE': mae,
        'RMSE': rmse
    })

    print("=" * 50)
    print(name)
    print("R2   :", r2)
    print("MAE  :", mae)
    print("RMSE :", rmse)


# ==========================================
# LINEAR REGRESSION
# ==========================================
evaluate_model(
    'Linear Regression',
    LinearRegression()
)


# ==========================================
# POLYNOMIAL DEGREE 2
# ==========================================
poly2 = Pipeline([
    (
        'poly',
        PolynomialFeatures(
            degree=2,
            include_bias=False
        )
    ),
    (
        'lr',
        LinearRegression()
    )
])

evaluate_model(
    'Polynomial Degree 2',
    poly2
)


# ==========================================
# POLYNOMIAL DEGREE 3
# ==========================================
poly3 = Pipeline([
    (
        'poly',
        PolynomialFeatures(
            degree=3,
            include_bias=False
        )
    ),
    (
        'lr',
        LinearRegression()
    )
])

evaluate_model(
    'Polynomial Degree 3',
    poly3
)


# ==========================================
# RANDOM FOREST
# ==========================================
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

evaluate_model(
    'Random Forest',
    rf
)


# ==========================================
# XGBOOST
# ==========================================
if xgb_available:

    xgb = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    evaluate_model(
        'XGBoost',
        xgb
    )


# ==========================================
# HASIL AKHIR
# ==========================================
hasil_df = pd.DataFrame(hasil)

hasil_df = hasil_df.sort_values(
    by='R2',
    ascending=False
)

print("\n")
print("=" * 60)
print("HASIL AKHIR")
print("=" * 60)
print(hasil_df)

Linear Regression
R2   : -8.16338992336163
MAE  : 0.7392932013451099
RMSE : 0.7648100594429007
Polynomial Degree 2
R2   : -8.553995724696234
MAE  : 0.7547536740262747
RMSE : 0.7809406493724145
Polynomial Degree 3
R2   : -10.22729293703284
MAE  : 0.8178876609011883
RMSE : 0.8465703133640846
Random Forest
R2   : -15.896107886146652
MAE  : 1.0045753743256283
RMSE : 1.0385295178922886
XGBoost
R2   : -12.966118779743317
MAE  : 0.9162988597691529
RMSE : 0.9441986466168356


HASIL AKHIR
                 Model         R2       MAE      RMSE
0    Linear Regression  -8.163390  0.739293  0.764810
1  Polynomial Degree 2  -8.553996  0.754754  0.780941
2  Polynomial Degree 3 -10.227293  0.817888  0.846570
4              XGBoost -12.966119  0.916299  0.944199
3        Random Forest -15.896108  1.004575  1.038530
